# Driver Drowsiness Detection - Quick Start Guide

## Using the Pre-Trained Model

If you want to run the model **without training it again**, skip directly to the **last code block** in this notebook.

This notebook provides:
- A pre-trained Keras CNN model for **driver drowsiness detection**
- Haar cascade classifiers for **face and eye detection**
- **Live webcam feed** with real-time detection and drowsiness percentage display

## Instructions for Quick Start:

1. Ensure your webcam is connected and accessible
2. Navigate to the last code block in this notebook
3. Run the cell to start real-time detection
4. Press **'q'** or close the webcam window to exit

This allows you to test the model immediately without going through the training process.

## Setup and Dependencies

This section imports all required libraries for the drowsiness detection system.

**Required Libraries:**
- **TensorFlow/Keras**: Deep learning framework for building and training the CNN model
- **NumPy**: Numerical computing for array operations
- **Matplotlib**: Visualization of training metrics and sample images
- **OpenCV**: Computer vision library for webcam capture and face detection

In [1]:

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import os
import json
import zipfile

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)

2025-12-14 22:50:40.255209: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-14 22:50:40.291947: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-14 22:50:41.310259: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TensorFlow version: 2.20.0
NumPy version: 2.2.6


## Dataset Configuration

This notebook expects the Driver Drowsiness Dataset (DDD) to be available locally.

**Expected Dataset Location:**
```
data/Driver Drowsiness Dataset (DDD)/
```

**Dataset Structure:**
The dataset should contain two subdirectories:
- `Drowsy/`: Images of drowsy drivers (eyes closed or partially closed)
- `Non Drowsy/`: Images of alert drivers (eyes open)

If the dataset is not present, refer to the next section for download instructions.

In [2]:
# Cell 2: Paths & parameters

data_dir = "../data/Driver Drowsiness Dataset (DDD)"

img_height = 64
img_width = 64
batch_size = 32

## Dataset Download (Optional)

**Note:** This cell is commented out by default.

If you need to download the dataset from Kaggle:
1. Obtain your Kaggle API credentials from [kaggle.com/account](https://www.kaggle.com/account)
2. Uncomment the code in the cell below
3. Replace `YOUR_USERNAME_HERE` and `YOUR_KEY_HERE` with your actual Kaggle credentials
4. Run the cell once to download and extract the dataset

After the initial download, the dataset will be available locally and this step can be skipped.

In [3]:
"""
--------------------------------------------------------------------------------
KAGGLE DOWNLOAD CODE (COMMENTED OUT BY DEFAULT)
--------------------------------------------------------------------------------

import os
import json
import zipfile

# ----------------------------------------
# 1. INSERT YOUR KAGGLE TOKEN HERE
# ----------------------------------------
kaggle_token = {
    "username": "YOUR_USERNAME_HERE",
    "key": "YOUR_KEY_HERE"
}

home = os.path.expanduser("~")
kaggle_dir = os.path.join(home, ".kaggle")

# Create .kaggle folder if it doesn't exist
if not os.path.exists(kaggle_dir):
    os.makedirs(kaggle_dir)

# Write kaggle.json
kaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")
with open(kaggle_json_path, "w") as f:
    json.dump(kaggle_token, f)

# Fix permissions
os.chmod(kaggle_json_path, 0o600)

# ----------------------------------------
# 2. DOWNLOAD THE DATASET
# ----------------------------------------
print("Downloading Driver Drowsiness Dataset (DDD)...")
os.system("kaggle datasets download -d ismailnasri20/driver-drowsiness-dataset-ddd -p data")

# ----------------------------------------
# 3. UNZIP THE DATASET
# ----------------------------------------
zip_file = "data/driver-drowsiness-dataset-ddd.zip"

print("Extracting dataset...")
with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall("data")

print("Done! Dataset available in: data/")
--------------------------------------------------------------------------------

After running this once, the dataset will be available locally and the rest of 
the project will work normally.

================================================================================
"""

'\n--------------------------------------------------------------------------------\nKAGGLE DOWNLOAD CODE (COMMENTED OUT BY DEFAULT)\n--------------------------------------------------------------------------------\n\nimport os\nimport json\nimport zipfile\n\n# ----------------------------------------\n# 1. INSERT YOUR KAGGLE TOKEN HERE\n# ----------------------------------------\nkaggle_token = {\n    "username": "YOUR_USERNAME_HERE",\n    "key": "YOUR_KEY_HERE"\n}\n\nhome = os.path.expanduser("~")\nkaggle_dir = os.path.join(home, ".kaggle")\n\n# Create .kaggle folder if it doesn\'t exist\nif not os.path.exists(kaggle_dir):\n    os.makedirs(kaggle_dir)\n\n# Write kaggle.json\nkaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")\nwith open(kaggle_json_path, "w") as f:\n    json.dump(kaggle_token, f)\n\n# Fix permissions\nos.chmod(kaggle_json_path, 0o600)\n\n# ----------------------------------------\n# 2. DOWNLOAD THE DATASET\n# ----------------------------------------\nprint("Do

## Dataset Loading and Splitting

This section loads images from the dataset directory and creates training and validation datasets.

**Configuration:**
- **Validation Split**: 20% of data reserved for validation
- **Image Size**: 64x64 pixels (resized automatically)
- **Batch Size**: 32 images per batch
- **Seed**: 42 (for reproducible random splits)
`
The dataset is automatically split into:
- **Training Set**: 80% of images (used to train the model)
- **Validation Set**: 20% of images (used to evaluate model performance)

In [4]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

class_names = train_ds.class_names
print("Class names:", class_names)


FileNotFoundError: [Errno 2] No such file or directory: 'data/Driver Drowsiness Dataset (DDD)'

## Dataset Optimization and Visualization

**Performance Optimization:**
- **Caching**: Keeps dataset in memory after first epoch for faster subsequent epochs
- **Shuffling**: Randomizes training data order (buffer size: 1000)
- **Prefetching**: Prepares next batch while current batch is being processed

**Visualization:**
Displays a 3x3 grid of sample images from the training dataset with their corresponding labels. This helps verify that the dataset is loaded correctly and shows the variety of images in each class.

In [5]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# Visual inspection
plt.figure(figsize=(6, 6))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i+1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")


NameError: name 'train_ds' is not defined

## CNN Model Architecture

This section defines the Convolutional Neural Network (CNN) architecture for binary classification.

**Model Components:**

1. **Data Augmentation Layer:**
   - Random horizontal flips
   - Random rotation (10% range)
   - Random contrast adjustment (10% range)
   - Helps prevent overfitting by creating variations of training images

2. **Preprocessing:**
   - Pixel value normalization (0-1 range)

3. **Convolutional Layers:**
   - Conv2D Layer 1: 32 filters, 3x3 kernel, ReLU activation
   - MaxPooling2D: 2x2 pool size
   - Conv2D Layer 2: 64 filters, 3x3 kernel, ReLU activation
   - MaxPooling2D: 2x2 pool size
   - Conv2D Layer 3: 128 filters, 3x3 kernel, ReLU activation
   - MaxPooling2D: 2x2 pool size

4. **Dense Layers:**
   - Flatten: Converts 3D feature maps to 1D
   - Dropout: 50% dropout rate to prevent overfitting
   - Dense Layer: 128 neurons, ReLU activation
   - Output Layer: 1 neuron, sigmoid activation (binary classification)

**Total Parameters:** ~1.14 million trainable parameters

In [6]:
num_classes = len(class_names)

data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomContrast(0.1),
    ]
)

inputs = keras.Input(shape=(img_height, img_width, 3))

x = data_augmentation(inputs)
x = layers.Rescaling(1./255)(x)

# Convolutional layers
x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
x = layers.MaxPooling2D()(x)

# Dense layers
x = layers.Flatten()(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(128, activation="relu")(x)

# Output layer: binary classification
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)

model.summary()


NameError: name 'class_names' is not defined

## Model Compilation and Training

**Compilation Configuration:**
- **Optimizer**: Adam (adaptive learning rate optimization)
- **Loss Function**: Binary crossentropy (suitable for binary classification)
- **Metrics**: Accuracy

**Training Configuration:**
- **Epochs**: 10 (can be increased to 15 for potentially better results)
- **Training Time**: Approximately 30-40 seconds per epoch (hardware dependent)

The model will train on the training dataset and validate on the validation dataset after each epoch. Training history (accuracy and loss) is stored for visualization.

In [7]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Training
epochs = 10  # you can increase to 15 if training is fast

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs
)


NameError: name 'model' is not defined

## Training Performance Visualization

This section plots the training and validation metrics over all epochs.

**Graphs Generated:**

1. **Accuracy Plot:**
   - Shows training and validation accuracy trends
   - Helps identify if the model is learning effectively
   - Validation accuracy close to training accuracy indicates good generalization

2. **Loss Plot:**
   - Shows training and validation loss trends
   - Decreasing loss indicates the model is improving
   - Large gap between training and validation loss may indicate overfitting

**Expected Results:**
- High accuracy (>95%) on both training and validation sets
- Low and decreasing loss values
- Minimal overfitting due to data augmentation and dropout

In [8]:
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]

epochs_range = range(len(acc))

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label="Train Acc")
plt.plot(epochs_range, val_acc, label="Val Acc")
plt.legend()
plt.title("Accuracy")

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label="Train Loss")
plt.plot(epochs_range, val_loss, label="Val Loss")
plt.legend()
plt.title("Loss")

plt.show()


NameError: name 'history' is not defined

## Model Persistence

Saves the trained CNN model to disk for later use in real-time detection.

**Save Location:** `../models/drowsiness_cnn.h5`

**File Format:** HDF5 format (.h5)
- Contains model architecture, weights, and training configuration
- Can be loaded for inference without retraining
- Portable across different systems with TensorFlow installed

In [9]:
model.save("../models/drowsiness_cnn.h5")
print("Model saved as ../models/drowsiness_cnn.h5")

NameError: name 'model' is not defined

## Real-Time Detection - Model Loading

Loads the pre-trained CNN model from disk for real-time drowsiness detection.

**Model Path:** `../models/drowsiness_cnn.h5`

**Class Labels:**
- Index 0: 'Drowsy' - Eyes closed or partially closed
- Index 1: 'Non Drowsy' - Eyes fully open and alert

The model outputs a probability value between 0 and 1 (sigmoid activation), where values closer to 0 indicate drowsiness and values closer to 1 indicate alertness.

In [10]:
# Cell 9: Real-time detection setup

import cv2
import numpy as np
from tensorflow import keras

# Load trained model from models directory
realtime_model = keras.models.load_model("../models/drowsiness_cnn.h5")

# Class names (same order as training)
class_names = ['Drowsy', 'Non Drowsy']

print("Model loaded successfully!")

E0000 00:00:1765749062.300898   47303 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1765749062.307669   47303 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model loaded successfully!


## Prediction Helper Functions

Utility functions for preprocessing eye images and generating drowsiness predictions.

**Functions Defined:**

1. **preprocess_eye(eye_img, img_height=64, img_width=64)**
   - Resizes eye image to model input size (64x64)
   - Converts BGR to RGB color space
   - Normalizes pixel values to 0-1 range
   - Adds batch dimension for model input
   - Returns preprocessed image array ready for prediction

2. **predict_drowsiness(eye_img)**
   - Takes raw eye image as input
   - Applies preprocessing
   - Runs model inference
   - Returns drowsiness probability (0 = drowsy, 1 = alert)
   - Float value between 0.0 and 1.0

In [ ]:
def preprocess_eye(eye_img, img_height=64, img_width=64):
    """Preprocess eye image for model prediction."""
    eye_resized = cv2.resize(eye_img, (img_width, img_height))
    eye_rgb = cv2.cvtColor(eye_resized, cv2.COLOR_BGR2RGB)
    eye_array = eye_rgb.astype("float32") / 255.0
    return np.expand_dims(eye_array, axis=0)

def predict_drowsiness(eye_img):
    """Return probability of drowsiness (0 to 1)."""
    x = preprocess_eye(eye_img)
    prob = float(realtime_model.predict(x, verbose=0)[0][0])  # sigmoid output
    return prob

## Real-Time Webcam Drowsiness Detection (Alternative Implementation)

This cell provides an alternative implementation using Haar cascades for face and eye detection.

**Detection Pipeline:**
1. Captures frames from webcam
2. Converts to grayscale for face detection
3. Detects faces using Haar cascade classifier
4. Detects eyes within each face region
5. Preprocesses and classifies each eye image
6. Tracks consecutive drowsy frames
7. Displays visual alert if drowsiness threshold is exceeded

**Configuration Parameters:**
- **DROWSY_THRESHOLD**: 0.60 - Probability threshold for drowsiness classification
- **FRAMES_THRESHOLD**: 15 - Number of consecutive drowsy frames before alarm triggers

**Controls:**
- Press **'q'** to quit
- Close the window to exit

**Note:** This cell is currently commented out. Remove the triple quotes to enable this implementation.

In [ ]:
'''
# Cell 11: Webcam detection

# Load Haar cascades
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)
eye_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_eye.xml"
)

cap = cv2.VideoCapture(0)

# Drowsiness logic parameters
DROWSY_THRESHOLD = 0.60       # probability above this = drowsy
FRAMES_THRESHOLD = 15         # alarm after these many consecutive drowsy frames
drowsy_frames = 0

print("Starting webcam... Press 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    display = frame.copy()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Detect faces
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    is_drowsy_this_frame = False

    for (x, y, w, h) in faces:
        # Draw rectangle around face
        cv2.rectangle(display, (x, y), (x+w, y+h), (255, 255, 0), 2)

        roi_gray = gray[y:y+h, x:x+w]
        roi_color = frame[y:y+h, x:x+w]

        eyes = eye_cascade.detectMultiScale(roi_gray, 1.1, 3)

        for (ex, ey, ew, eh) in eyes:
            eye_img = roi_color[ey:ey+eh, ex:ex+ew]
            prob = predict_drowsiness(eye_img)

            label = f"Drowsy: {prob:.2f}"
            color = (0, 0, 255) if prob >= DROWSY_THRESHOLD else (0, 255, 0)

            # Draw rectangle for eye
            cv2.rectangle(roi_color, (ex, ey), (ex+ew, ey+eh), color, 2)
            cv2.putText(roi_color, label, (ex, ey - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
            
            if prob >= DROWSY_THRESHOLD:
                is_drowsy_this_frame = True
        
        break  # use only first face

    # track consecutive drowsy frames
    if is_drowsy_this_frame:
        drowsy_frames += 1
    else:
        drowsy_frames = 0

    # Trigger alarm visually
    if drowsy_frames >= FRAMES_THRESHOLD:
        cv2.putText(display, "WAKE UP! YOU ARE DROWSY!",
                    (50, 80), cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                    (0, 0, 255), 3)

    cv2.imshow("Drowsiness Detection", display)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
'''

Starting webcam... Press 'q' to quit.


In [ ]:
# ESSENTIAL IMPORTS - Run this first after every kernel restart

import numpy as np
import cv2
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

print("Imports loaded successfully.")

In [ ]:
realtime_model = keras.models.load_model("../models/drowsiness_cnn.h5")

In [ ]:
# Cell 9: Real-time detection setup
import cv2

# Reload the trained model from models directory
realtime_model = keras.models.load_model("../models/drowsiness_cnn.h5")

# For binary classification we will assume:
#   0 -> class_names[0]
#   1 -> class_names[1]
print("Class names:", class_names)

## Real-Time Driver Drowsiness Detection System

**Main Detection Loop Implementation**

This is the primary real-time detection system that monitors driver alertness through webcam feed.

**System Components:**

1. **Model Loading:**
   - Loads pre-trained model from `../models/drowsiness_model.keras`
   - Input size: 224x224 pixels
   - Binary classification output

2. **Face Detection:**
   - Uses Haar cascade classifier for frontal face detection
   - Optimized parameters:
     - Scale factor: 1.1
     - Min neighbors: 6
     - Min face size: 60x60 pixels

3. **Drowsiness Detection Logic:**
   - Processes detected face region through CNN model
   - Calculates drowsiness percentage from model probability
   - Tracks consecutive drowsy frames
   - Triggers alert after 15 consecutive drowsy frames

4. **Visual Feedback:**
   - Green rectangle and text: Driver is alert
   - Red rectangle and text: Driver is drowsy
   - Displays real-time drowsiness percentage
   - Alert message: "DROWSY! WAKE UP!" when threshold exceeded
   - Yellow text: No face detected

**Controls:**
- **Close window**: Click the X button to exit
- **Press 'q'**: Alternative method to quit

**Performance Notes:**
- Processes frames in real-time (video feed dependent on webcam FPS)
- Window-driven loop for better stability
- Graceful cleanup on exit

In [ ]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# =========================
# Load Drowsiness Model
# =========================
MODEL_PATH = "../models/drowsiness_model.keras"
model = load_model(MODEL_PATH)
IMG_SIZE = (224, 224)

# =========================
# Haar cascade for face detection
# =========================
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

# =========================
# Drowsiness logic
# =========================
DROWSY_FRAMES_THRESHOLD = 15
drowsy_frame_count = 0

# =========================
# Webcam
# =========================
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Error: Webcam not found.")
    exit()

window_name = "Driver Drowsiness Detector"
cv2.namedWindow(window_name)

# =========================
# Main loop (window-driven)
# =========================
while cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) >= 1:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Improved face detection
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=6,
        minSize=(60, 60)
    )

    status_text = "Awake"
    drowsiness_percent = 0

    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

        roi = frame[y:y+h, x:x+w]
        resized = cv2.resize(roi, IMG_SIZE)
        normalized = resized / 255.0
        reshaped = np.expand_dims(normalized, axis=0)

        prob = float(model.predict(reshaped, verbose=0)[0][0])
        drowsiness_percent = (1 - prob) * 100

        if prob < 0.5:
            drowsy_frame_count += 1
        else:
            drowsy_frame_count = 0

        if drowsy_frame_count >= DROWSY_FRAMES_THRESHOLD:
            status_text = "DROWSY! WAKE UP!"
            color = (0, 0, 255)
        else:
            status_text = "Awake"
            color = (0, 255, 0)

        cv2.putText(frame, f"{status_text} ({drowsiness_percent:.1f}%)",
                    (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 3)

    if len(faces) == 0:
        cv2.putText(frame, "No face detected", (50, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)

    cv2.imshow(window_name, frame)

    # Also allow quitting with 'q' key
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()